In [ ]:
import pandas as pd

orders = pd.read_csv('../raw/orders.csv')
orders.head()

In [ ]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv(override=True)
db_url = os.getenv("DATABASE_URL")

engine = create_engine(db_url)

print("Connected! Starting migration...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=5000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=5000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=5000)
print("departments done")

orders.to_sql('orders', engine, if_exists='replace', index=False, chunksize=5000)
print("orders done")

order_products_full.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=5000)
print("order_products_full done — Migration complete!")

In [ ]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv(override=True)
db_url = os.getenv("DATABASE_URL")

engine = create_engine(db_url)

print("Connected! Starting migration...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=5000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=5000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=5000)
print("departments done")

orders.to_sql('orders', engine, if_exists='replace', index=False, chunksize=5000)
print("orders done")

order_products_full.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=5000)
print("order_products_full done — Migration complete!")

In [ ]:
import pandas as pd
import sqlite3
import os
from dotenv import load_dotenv
from google import genai
from sqlalchemy import create_engine

# Load raw data
base_path = r'C:\Projects\Insights Agent\raw'
orders = pd.read_csv(f'{base_path}\\orders.csv')
products = pd.read_csv(f'{base_path}\\products.csv')
aisles = pd.read_csv(f'{base_path}\\aisles.csv')
departments = pd.read_csv(f'{base_path}\\departments.csv')
order_products_train = pd.read_csv(f'{base_path}\\order_products__train.csv')

order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

# SQLite connection (local)
conn = sqlite3.connect(r'C:\Projects\Insights Agent\insights_agent.db')

# Environment variables
load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")
db_url = os.getenv("DATABASE_URL")

# Gemini client
client = genai.Client(api_key=api_key)

print("Master setup complete! All variables ready.")

In [ ]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv(override=True)
db_url = os.getenv("DATABASE_URL")

engine = create_engine(db_url)

print("Connected! Starting migration...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=5000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=5000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=5000)
print("departments done")

orders.to_sql('orders', engine, if_exists='replace', index=False, chunksize=5000)
print("orders done")

order_products_full.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=5000)
print("order_products_full done — Migration complete!")

In [ ]:
import pandas as pd
import sqlite3

base_path = r'C:\Projects\Insights Agent\raw'

orders = pd.read_csv(f'{base_path}\\orders.csv')
products = pd.read_csv(f'{base_path}\\products.csv')
aisles = pd.read_csv(f'{base_path}\\aisles.csv')
departments = pd.read_csv(f'{base_path}\\departments.csv')
order_products_train = pd.read_csv(f'{base_path}\\order_products__train.csv')

order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

print("Data loaded and merged!")

In [ ]:
conn = sqlite3.connect(r'C:\Projects\Insights Agent\insights_agent.db')

order_products_full.to_sql('order_products_full', conn, if_exists='replace', index=False, chunksize=10000, method='multi')
orders.to_sql('orders', conn, if_exists='replace', index=False, chunksize=10000, method='multi')
products.to_sql('products', conn, if_exists='replace', index=False)
aisles.to_sql('aisles', conn, if_exists='replace', index=False)
departments.to_sql('departments', conn, if_exists='replace', index=False)

print("Tables recreated successfully!")

In [ ]:
order_products_full.to_sql('order_products_full', conn, if_exists='replace', index=False, chunksize=10000)
orders.to_sql('orders', conn, if_exists='replace', index=False, chunksize=10000)
products.to_sql('products', conn, if_exists='replace', index=False)
aisles.to_sql('aisles', conn, if_exists='replace', index=False)
departments.to_sql('departments', conn, if_exists='replace', index=False)

print("Tables recreated successfully!")

In [ ]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

In [ ]:
generated_sql = """
SELECT
  T1.department
FROM departments AS T1
JOIN order_products_full AS T2
  ON T1.department_id = T2.department_id
GROUP BY
  T1.department_id
ORDER BY
  AVG(T2.reordered) DESC
LIMIT 1;
"""

result = pd.read_sql_query(generated_sql, conn)
print(result)

In [ ]:
import pandas as pd
import sqlite3
import os
from dotenv import load_dotenv
from google import genai

conn = sqlite3.connect(r'C:\Projects\Insights Agent\insights_agent.db')
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Setup complete!")

In [ ]:
schema_info = """
Tables:
1. order_products_full(order_id, product_id, add_to_cart_order, reordered, product_name, aisle_id, department_id, aisle, department)
2. orders(order_id, user_id, eval_set, order_number, order_dow, order_hour_of_day, days_since_prior_order)
3. products(product_id, product_name, aisle_id, department_id)
4. aisles(aisle_id, aisle)
5. departments(department_id, department)
"""

def ask_question(question):
    # Step 1: Gemini se SQL generate karwao
    prompt = f"""You are a SQL expert. Given this database schema:
{schema_info}

Convert this question into a valid SQLite query. Only return the SQL query, nothing else. No markdown, no explanation, no code fences.

Question: {question}
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    
    sql_query = response.text.strip()
    # Kabhi kabhi Gemini markdown fences add kar deta hai, unhe clean karo
    sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
    
    print("Generated SQL:\n", sql_query)
    
    # Step 2: Database pe run karo
    try:
        result = pd.read_sql_query(sql_query, conn)
        return result
    except Exception as e:
        return f"Error running SQL: {e}"

# Test karo
ask_question("Which department has the highest reorder rate?")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")
print("Key starts with:", api_key[:10] if api_key else "NOT FOUND")
print("Key length:", len(api_key) if api_key else 0)

In [ ]:
from google import genai

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say hello"
)
print(response.text)

In [ ]:
schema_info = """
Tables:
1. order_products_full(order_id, product_id, add_to_cart_order, reordered, product_name, aisle_id, department_id, aisle, department)
2. orders(order_id, user_id, eval_set, order_number, order_dow, order_hour_of_day, days_since_prior_order)
3. products(product_id, product_name, aisle_id, department_id)
4. aisles(aisle_id, aisle)
5. departments(department_id, department)
"""

def ask_question(question):
    prompt = f"""You are a SQL expert. Given this database schema:
{schema_info}

Convert this question into a valid SQLite query. Only return the SQL query, nothing else. No markdown, no explanation, no code fences.

Question: {question}
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    
    sql_query = response.text.strip()
    sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
    
    print("Generated SQL:\n", sql_query)
    
    try:
        result = pd.read_sql_query(sql_query, conn)
        return result
    except Exception as e:
        return f"Error running SQL: {e}"

# Test karo
ask_question("Which department has the highest reorder rate?")

In [ ]:
import pandas as pd
import sqlite3

base_path = r'C:\Projects\Insights Agent\raw'

orders = pd.read_csv(f'{base_path}\\orders.csv')
products = pd.read_csv(f'{base_path}\\products.csv')
aisles = pd.read_csv(f'{base_path}\\aisles.csv')
departments = pd.read_csv(f'{base_path}\\departments.csv')
order_products_train = pd.read_csv(f'{base_path}\\order_products__train.csv')

order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

conn = sqlite3.connect(r'C:\Projects\Insights Agent\insights_agent.db')

order_products_full.to_sql('order_products_full', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
aisles.to_sql('aisles', conn, if_exists='replace', index=False)
departments.to_sql('departments', conn, if_exists='replace', index=False)

print("Tables recreated successfully!")

In [4]:
import pandas as pd
import sqlite3
import os
from dotenv import load_dotenv
from google import genai
from sqlalchemy import create_engine

# Load raw data
base_path = r'C:\Projects\Insights Agent\raw'
orders = pd.read_csv(f'{base_path}\\orders.csv')
products = pd.read_csv(f'{base_path}\\products.csv')
aisles = pd.read_csv(f'{base_path}\\aisles.csv')
departments = pd.read_csv(f'{base_path}\\departments.csv')
order_products_train = pd.read_csv(f'{base_path}\\order_products__train.csv')

order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

# SQLite connection (local)
conn = sqlite3.connect(r'C:\Projects\Insights Agent\insights_agent.db')

# Environment variables
load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")
db_url = os.getenv("DATABASE_URL")

# Gemini client
client = genai.Client(api_key=api_key)

print("Master setup complete! All variables ready.")

Master setup complete! All variables ready.


In [5]:
engine = create_engine(db_url)

print("Starting migration...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=5000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=5000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=5000)
print("departments done")

orders.to_sql('orders', engine, if_exists='replace', index=False, chunksize=5000)
print("orders done")

order_products_full.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=5000)
print("order_products_full done — Migration complete!")

Starting migration...


OperationalError: (psycopg2.OperationalError) connection to server at "aws-0-ap-southeast-2.pooler.supabase.com" (3.106.102.114), port 5432 failed: FATAL:  password authentication failed for user "postgres"
connection to server at "aws-0-ap-southeast-2.pooler.supabase.com" (3.106.102.114), port 5432 failed: FATAL:  password authentication failed for user "postgres"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [1]:
import pandas as pd
import sqlite3
import os
from dotenv import load_dotenv
from google import genai
from sqlalchemy import create_engine

# Load raw data
base_path = r'C:\Projects\Insights Agent\raw'
orders = pd.read_csv(f'{base_path}\\orders.csv')
products = pd.read_csv(f'{base_path}\\products.csv')
aisles = pd.read_csv(f'{base_path}\\aisles.csv')
departments = pd.read_csv(f'{base_path}\\departments.csv')
order_products_train = pd.read_csv(f'{base_path}\\order_products__train.csv')

order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

# SQLite connection (local)
conn = sqlite3.connect(r'C:\Projects\Insights Agent\insights_agent.db')

# Environment variables
load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")
db_url = os.getenv("DATABASE_URL")

# Gemini client
client = genai.Client(api_key=api_key)

print("Master setup complete! All variables ready.")

Master setup complete! All variables ready.


In [2]:
engine = create_engine(db_url)

print("Starting migration...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=5000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=5000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=5000)
print("departments done")

orders.to_sql('orders', engine, if_exists='replace', index=False, chunksize=5000)
print("orders done")

order_products_full.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=5000)
print("order_products_full done — Migration complete!")

Starting migration...
products done
aisles done
departments done


PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

In [3]:
from sqlalchemy import create_engine

# Sample lo - migration ke liye kaafi hai
orders_sample = orders.sample(n=100000, random_state=42)
order_products_full_sample = order_products_full.sample(n=200000, random_state=42)

# Fresh engine banao
engine = create_engine(db_url)

print("Starting migration (sampled data)...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=2000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=2000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=2000)
print("departments done")

orders_sample.to_sql('orders', engine, if_exists='replace', index=False, chunksize=2000)
print("orders done (sampled - 100K rows)")

order_products_full_sample.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=2000)
print("order_products_full done (sampled - 200K rows) — Migration complete!")

Starting migration (sampled data)...
products done
aisles done
departments done
orders done (sampled - 100K rows)
order_products_full done (sampled - 200K rows) — Migration complete!


In [ ]:
from sqlalchemy import create_engine

# Sample lo - migration ke liye kaafi hai
orders_sample = orders.sample(n=100000, random_state=42)
order_products_full_sample = order_products_full.sample(n=200000, random_state=42)

# Fresh engine banao
engine = create_engine(db_url)

print("Starting migration (sampled data)...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=2000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=2000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=2000)
print("departments done")

orders_sample.to_sql('orders', engine, if_exists='replace', index=False, chunksize=2000)
print("orders done (sampled - 100K rows)")

order_products_full_sample.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=2000)
print("order_products_full done (sampled - 200K rows) — Migration complete!")

In [4]:
from sqlalchemy import create_engine

# Sample lo - migration ke liye kaafi hai
orders_sample = orders.sample(n=100000, random_state=42)
order_products_full_sample = order_products_full.sample(n=200000, random_state=42)

# Fresh engine banao
engine = create_engine(db_url)

print("Starting migration (sampled data)...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=2000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=2000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=2000)
print("departments done")

orders_sample.to_sql('orders', engine, if_exists='replace', index=False, chunksize=2000)
print("orders done (sampled - 100K rows)")

order_products_full_sample.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=2000)
print("order_products_full done (sampled - 200K rows) — Migration complete!")

Starting migration (sampled data)...
products done
aisles done
departments done
orders done (sampled - 100K rows)
order_products_full done (sampled - 200K rows) — Migration complete!


In [5]:
test_df = pd.read_sql_query("SELECT department, COUNT(*) as cnt FROM order_products_full GROUP BY department ORDER BY cnt DESC LIMIT 5", engine)
print(test_df)

   department    cnt
0     produce  59101
1  dairy eggs  31573
2      snacks  17173
3   beverages  16265
4      frozen  14481


In [ ]:
schema_info = """
Tables:
1. order_products_full(order_id, product_id, add_to_cart_order, reordered, product_name, aisle_id, department_id, aisle, department)
2. orders(order_id, user_id, eval_set, order_number, order_dow, order_hour_of_day, days_since_prior_order)
3. products(product_id, product_name, aisle_id, department_id)
4. aisles(aisle_id, aisle)
5. departments(department_id, department)
"""

def ask_question(question):
    prompt = f"""You are a SQL expert. Given this database schema:
{schema_info}

Convert this question into a valid SQLite query. Only return the SQL query, nothing else. No markdown, no explanation, no code fences.

Question: {question}
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    
    sql_query = response.text.strip()
    sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
    
    print("Generated SQL:\n", sql_query)
    
    try:
        result = pd.read_sql_query(sql_query, conn)
        return result
    except Exception as e:
        return f"Error running SQL: {e}"

# Test karo
ask_question("Which department has the highest reorder rate?")

In [ ]:
ask_question("What are the top 5 most ordered products?")

In [ ]:
ask_question("What is the average number of items per order?")

In [ ]:
def ask_question(question, explain=True):
    # Step 1: SQL generate karo
    prompt = f"""You are a SQL expert. Given this database schema:
{schema_info}

Convert this question into a valid SQLite query. Only return the SQL query, nothing else. No markdown, no explanation, no code fences.

Question: {question}
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    
    sql_query = response.text.strip()
    sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
    
    print("Generated SQL:\n", sql_query)
    
    # Step 2: Database pe run karo
    try:
        result = pd.read_sql_query(sql_query, conn)
    except Exception as e:
        return f"Error running SQL: {e}"
    
    print("\nRaw Result:\n", result)
    
    # Step 3: Result ko natural language mein explain karwao
    if explain:
        explain_prompt = f"""The user asked: "{question}"

The SQL query returned this result:
{result.to_string(index=False)}

Write a single, clear sentence answering the user's question based on this result. Be specific with numbers. Do not mention SQL or databases."""
        
        explanation = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=explain_prompt
        )
        
        print("\n💡 Answer:", explanation.text.strip())
    
    return result

In [ ]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv(override=True)
db_url = os.getenv("DATABASE_URL")

engine = create_engine(db_url)

print("Connected! Starting migration...")

# Chhoti tables pehle (fast)
products.to_sql('products', engine, if_exists='replace', index=False, chunksize=5000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=5000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=5000)
print("departments done")

orders.to_sql('orders', engine, if_exists='replace', index=False, chunksize=5000)
print("orders done")

# Sabse badi table last mein
order_products_full.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=5000)
print("order_products_full done — Migration complete!")

In [ ]:
import pandas as pd

base_path = r'C:\Projects\Insights Agent\raw'

orders = pd.read_csv(f'{base_path}\\orders.csv')
products = pd.read_csv(f'{base_path}\\products.csv')
aisles = pd.read_csv(f'{base_path}\\aisles.csv')
departments = pd.read_csv(f'{base_path}\\departments.csv')
order_products_train = pd.read_csv(f'{base_path}\\order_products__train.csv')

order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

print("Data loaded!")

In [ ]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv(override=True)
db_url = os.getenv("DATABASE_URL")

engine = create_engine(db_url)

print("Connected! Starting migration...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=5000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=5000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=5000)
print("departments done")

orders.to_sql('orders', engine, if_exists='replace', index=False, chunksize=5000)
print("orders done")

order_products_full.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=5000)
print("order_products_full done — Migration complete!")

In [ ]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv(override=True)
db_url = os.getenv("DATABASE_URL")

engine = create_engine(db_url)

print("Connected! Starting migration...")

products.to_sql('products', engine, if_exists='replace', index=False, chunksize=5000)
print("products done")

aisles.to_sql('aisles', engine, if_exists='replace', index=False, chunksize=5000)
print("aisles done")

departments.to_sql('departments', engine, if_exists='replace', index=False, chunksize=5000)
print("departments done")

orders.to_sql('orders', engine, if_exists='replace', index=False, chunksize=5000)
print("orders done")

order_products_full.to_sql('order_products_full', engine, if_exists='replace', index=False, chunksize=5000)
print("order_products_full done — Migration complete!")

In [ ]:
ask_question("Which department has the highest reorder rate?")

In [ ]:
ask_question("Which day of the week has the most orders?")

In [ ]:
import pandas as pd
import sqlite3

base_path = r'C:\Projects\Insights Agent\raw'

orders = pd.read_csv(f'{base_path}\\orders.csv')
products = pd.read_csv(f'{base_path}\\products.csv')
aisles = pd.read_csv(f'{base_path}\\aisles.csv')
departments = pd.read_csv(f'{base_path}\\departments.csv')
order_products_train = pd.read_csv(f'{base_path}\\order_products__train.csv')

order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

conn = sqlite3.connect(r'C:\Projects\Insights Agent\insights_agent.db')

order_products_full.to_sql('order_products_full', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
aisles.to_sql('aisles', conn, if_exists='replace', index=False)
departments.to_sql('departments', conn, if_exists='replace', index=False)

print("Tables recreated successfully!")

In [ ]:
generated_sql = """
SELECT
  T1.department
FROM departments AS T1
JOIN order_products_full AS T2
  ON T1.department_id = T2.department_id
GROUP BY
  T1.department_id
ORDER BY
  AVG(T2.reordered) DESC
LIMIT 1;
"""

result = pd.read_sql_query(generated_sql, conn)
print(result)

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../insights_agent.db')

In [ ]:
import pandas as pd
import sqlite3
import os
from dotenv import load_dotenv
from google import genai

conn = sqlite3.connect('../insights_agent.db')
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Setup complete!")

In [ ]:
generated_sql = """
SELECT
  T1.department
FROM departments AS T1
JOIN order_products_full AS T2
  ON T1.department_id = T2.department_id
GROUP BY
  T1.department_id
ORDER BY
  AVG(T2.reordered) DESC
LIMIT 1;
"""

result = pd.read_sql_query(generated_sql, conn)
print(result)

In [ ]:
import os
print(os.path.abspath('../insights_agent.db'))
print(os.path.exists('../insights_agent.db'))

In [ ]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

In [ ]:
import pandas as pd
import sqlite3

# Raw files load karo
orders = pd.read_csv('../raw/orders.csv')
products = pd.read_csv('../raw/products.csv')
aisles = pd.read_csv('../raw/aisles.csv')
departments = pd.read_csv('../raw/departments.csv')
order_products_train = pd.read_csv('../raw/order_products__train.csv')

# Merge karke full table banao
order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

# Database mein daalo
conn = sqlite3.connect('../insights_agent.db')

order_products_full.to_sql('order_products_full', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
aisles.to_sql('aisles', conn, if_exists='replace', index=False)
departments.to_sql('departments', conn, if_exists='replace', index=False)

print("Tables recreated successfully!")

In [ ]:
import os
print(os.getcwd())

In [ ]:
import pandas as pd
import sqlite3

base_path = r'C:\Users\lenovo\OneDrive\Desktop\Insights Agent\raw'

orders = pd.read_csv(f'{base_path}\\orders.csv')
products = pd.read_csv(f'{base_path}\\products.csv')
aisles = pd.read_csv(f'{base_path}\\aisles.csv')
departments = pd.read_csv(f'{base_path}\\departments.csv')
order_products_train = pd.read_csv(f'{base_path}\\order_products__train.csv')

order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

conn = sqlite3.connect(r'C:\Users\lenovo\OneDrive\Desktop\Insights Agent\insights_agent.db')

order_products_full.to_sql('order_products_full', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
aisles.to_sql('aisles', conn, if_exists='replace', index=False)
departments.to_sql('departments', conn, if_exists='replace', index=False)

print("Tables recreated successfully!")

In [ ]:
import pandas as pd
import sqlite3

base_path = r'C:\Users\lenovo\OneDrive\Desktop\Insights Agent\raw'

orders = pd.read_csv(f'{base_path}\\orders.csv')
products = pd.read_csv(f'{base_path}\\products.csv')
aisles = pd.read_csv(f'{base_path}\\aisles.csv')
departments = pd.read_csv(f'{base_path}\\departments.csv')
order_products_train = pd.read_csv(f'{base_path}\\order_products__train.csv')

order_products_full = order_products_train.merge(products, on='product_id', how='left')
order_products_full = order_products_full.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')

conn = sqlite3.connect(r'C:\Users\lenovo\OneDrive\Desktop\Insights Agent\insights_agent.db')

order_products_full.to_sql('order_products_full', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
aisles.to_sql('aisles', conn, if_exists='replace', index=False)
departments.to_sql('departments', conn, if_exists='replace', index=False)

print("Tables recreated successfully!")

In [ ]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

model = genai.GenerativeModel("gemini-2.5-flash")
response = model.generate_content("Say hello in one short sentence.")

print(response.text)

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
print("Key loaded:", api_key is not None)

client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one short sentence."
)
print(response.text)

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
print("Key loaded:", api_key is not None)

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say hello in one short sentence."
)

print(response.text)

In [ ]:
schema_info = """
Tables:
1. order_products_full(order_id, product_id, add_to_cart_order, reordered, product_name, aisle_id, department_id, aisle, department)
2. orders(order_id, user_id, eval_set, order_number, order_dow, order_hour_of_day, days_since_prior_order)
3. products(product_id, product_name, aisle_id, department_id)
4. aisles(aisle_id, aisle)
5. departments(department_id, department)
"""

user_question = "Which department has the highest reorder rate?"

prompt = f"""You are a SQL expert. Given this database schema:
{schema_info}

Convert this question into a valid SQLite query. Only return the SQL query, nothing else.

Question: {user_question}
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)

In [ ]:
orders.shape

In [ ]:
orders.info()

In [ ]:
orders[orders['days_since_prior_order'].isnull()]['order_number'].unique()

In [ ]:
orders[orders['days_since_prior_order'].isnull()]['order_number'].unique()

In [ ]:
products = pd.read_csv('../raw/products.csv')
aisles = pd.read_csv('../raw/aisles.csv')
departments = pd.read_csv('../raw/departments.csv')
order_products_prior = pd.read_csv('../raw/order_products__prior.csv')
order_products_train = pd.read_csv('../raw/order_products__train.csv')

In [ ]:
print("products:", products.shape)
print("aisles:", aisles.shape)
print("departments:", departments.shape)
print("order_products_prior:", order_products_prior.shape)
print("order_products_train:", order_products_train.shape)

In [ ]:
order_products_train.head()

In [ ]:
order_products_train_named = order_products_train.merge(products, on='product_id', how='left')
order_products_train_named.head()

In [ ]:
order_products_full = order_products_train_named.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')
order_products_full.head()

In [ ]:
order_products_full.to_csv('../raw/order_products_full.csv', index=False)

In [ ]:
order_products_full['department'].value_counts()

In [ ]:
order_products_full.groupby('department')['reordered'].mean().sort_values(ascending=False)

In [ ]:
from scipy.stats import chi2_contingency

# Sirf dono departments ka data filter karo
subset = order_products_full[order_products_full['department'].isin(['dairy eggs', 'personal care'])]

# Contingency table banao (department vs reordered ka cross-tab)
contingency_table = pd.crosstab(subset['department'], subset['reordered'])
print(contingency_table)

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)

In [ ]:
from scipy.stats import ttest_ind

produce_cart_order = order_products_full[order_products_full['department'] == 'produce']['add_to_cart_order']
snacks_cart_order = order_products_full[order_products_full['department'] == 'snacks']['add_to_cart_order']

print("Produce average position:", produce_cart_order.mean())
print("Snacks average position:", snacks_cart_order.mean())

t_stat, p_value = ttest_ind(produce_cart_order, snacks_cart_order)
print("T-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
from scipy.stats import f_oneway

departments_to_compare = ['produce', 'dairy eggs', 'snacks', 'beverages', 'frozen']

groups = [
    order_products_full[order_products_full['department'] == dept]['add_to_cart_order']
    for dept in departments_to_compare
]

f_stat, p_value = f_oneway(*groups)

print("F-statistic:", f_stat)
print("p-value:", p_value)

# Har department ka average bhi dekh lete hain reference ke liye
for dept in departments_to_compare:
    avg = order_products_full[order_products_full['department'] == dept]['add_to_cart_order'].mean()
    print(f"{dept}: {avg:.2f}")

## Statistical Insights — Phase 4

### 1. Department vs Reorder Behavior (Chi-Square Test)
**Question:** Kya reorder rate department ke basis pe significantly differ karta hai?
- Dairy Eggs: 67.5% reorder rate | Personal Care: 33.7% reorder rate
- Chi-square = 9773.53, p < 0.001 → **Statistically significant**
- **Insight:** Fresh/perishable categories (dairy, produce) habit-driven hain; personal care items infrequent/planned purchases hain.

### 2. Cart Position — Produce vs Snacks (T-Test)
**Question:** Kya produce items snacks se pehle cart mein add hote hain?
- Produce avg position: 8.43 | Snacks avg position: 9.56
- T-statistic = -47.88, p < 0.001 → **Statistically significant**
- **Insight:** Essential/planned items pehle add hote hain, discretionary items baad mein.

### 3. Cart Position Across 5 Departments (ANOVA)
**Question:** Kya cart position multiple departments mein differ karta hai?
- Beverages: 7.14 | Dairy Eggs: 7.88 | Produce: 8.43 | Frozen: 9.44 | Snacks: 9.56
- F-statistic = 2494.29, p < 0.001 → **Statistically significant**
- **Insight:** Beverages/Dairy = planned purchases (early cart); Snacks/Frozen = impulse/secondary purchases (late cart).

### Business Takeaway
Data confirms two behavioral patterns: (1) purchase frequency/loyalty varies strongly by category type — fresh/perishable > discretionary; (2) shopping planning sequence follows a "essentials-first" pattern. These insights could inform personalization features — e.g., reminder nudges for high-reorder categories, or cart-suggestion ordering aligned with natural shopping sequence.

In [ ]:
import sqlite3

# Database connection banao (agar file exist nahi karti, naye banegi)
conn = sqlite3.connect('../insights_agent.db')

# Cleaned/merged table ko database mein daalo
order_products_full.to_sql('order_products_full', conn, if_exists='replace', index=False)

# Baaki core tables bhi daal do
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
aisles.to_sql('aisles', conn, if_exists='replace', index=False)
departments.to_sql('departments', conn, if_exists='replace', index=False)

print("Database created successfully!")

In [ ]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

In [ ]:
query = """
SELECT department, COUNT(*) as total_orders, AVG(reordered) as reorder_rate
FROM order_products_full
GROUP BY department
ORDER BY reorder_rate DESC
LIMIT 5
"""

result = pd.read_sql_query(query, conn)
print(result)

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()  # .env file se key load karega

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Say hello in one short sentence."}
    ]
)

print(response.content[0].text)